# Credit Card Fraud Detection (Project 19 - Ensemble Learning)
**Problem:** flag fraudulent card transactions under severe class imbalance (~0.17% fraud).
**Users:** a bank's fraud-review team. Model output is a *decision-support* score, not a final verdict.
**Plan:** baseline -> Decision Tree -> Logistic Regression -> Random Forest -> Boosting -> Voting; compare on val, report once on test.

In [ ]:
import time, warnings, sys
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, precision_recall_curve,
                             ConfusionMatrixDisplay)
warnings.filterwarnings("ignore")
RS = 42
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

## 1. Load data
Real data: Kaggle "Credit Card Fraud Detection" (ULB), file `creditcard.csv`, 284,807 rows,
columns Time, V1-V28 (PCA-anonymised), Amount, Class (1 = fraud). Put it next to this notebook.
If the file is missing, a synthetic stand-in is generated so the code can still be tested.

In [ ]:
def load_data(path="creditcard.csv", synthetic=False):
    if not synthetic:
        try:
            return pd.read_csv(path), "real"
        except FileNotFoundError:
            print("creditcard.csv not found -> using SYNTHETIC data (for testing only)")
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=40000, n_features=28, n_informative=12,
                               weights=[0.99, 0.01], class_sep=1.2, flip_y=0.002, random_state=RS)
    df = pd.DataFrame(X, columns=[f"V{i}" for i in range(1, 29)])
    rng = np.random.default_rng(RS)
    df["Time"] = np.sort(rng.integers(0, 172800, len(df)))
    df["Amount"] = np.round(rng.lognormal(3.5, 1.2, len(df)), 2)
    df["Class"] = y
    return df[["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]], "synthetic"

df, source = load_data(synthetic=False)
print(source, df.shape)

## 2. Data checks (documentation requirement)

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicates:", int(df.duplicated().sum()))
df = df.drop_duplicates().reset_index(drop=True)
vc = df["Class"].value_counts()
print(vc, f"\nFraud rate: {vc.get(1,0)/len(df):.4%}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
vc.plot.bar(ax=ax[0], title="Class counts"); 
df.loc[df.Class == 0, "Amount"].clip(upper=500).plot.hist(bins=50, ax=ax[1], alpha=.6, label="legit")
df.loc[df.Class == 1, "Amount"].clip(upper=500).plot.hist(bins=50, ax=ax[1], alpha=.6, label="fraud")
ax[1].legend(); ax[1].set_title("Amount (clipped at 500)")
plt.tight_layout(); plt.savefig("eda.png", dpi=120); plt.close()

## 3. Split (stratified 60/20/20) - no leakage
Scaling lives inside a Pipeline so it is fitted on training data only.
Resampling (SMOTE) is deliberately NOT applied before the split; we use class weights instead.

In [ ]:
X, y = df.drop(columns="Class"), df["Class"]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RS)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=RS)
print({k: (len(v), int(s.sum())) for k, v, s in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]})
prep = ColumnTransformer([("sc", RobustScaler(), ["Time", "Amount"])], remainder="passthrough")
pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

## 4. Models (baseline + individual + ensembles)

In [ ]:
def make(clf): return Pipeline([("prep", prep), ("clf", clf)])
models = {
    "Baseline (majority)": make(DummyClassifier(strategy="most_frequent")),
    "Logistic Regression": make(LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RS)),
    "Decision Tree": make(DecisionTreeClassifier(max_depth=6, class_weight="balanced", min_samples_leaf=10, random_state=RS)),
    "Random Forest": make(RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                                                 min_samples_leaf=2, n_jobs=-1, random_state=RS)),
}
if HAS_XGB:
    models["XGBoost"] = make(XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                             scale_pos_weight=pos_weight, eval_metric="aucpr", n_jobs=-1, random_state=RS))
else:
    models["Gradient Boosting (HistGB)"] = make(HistGradientBoostingClassifier(
        class_weight="balanced", max_iter=300, learning_rate=0.1, random_state=RS))
boost_name = [k for k in models if k.startswith(("XGB", "Gradient"))][0]
models["Voting (LR+RF+Boost)"] = make(VotingClassifier(
    [("lr", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RS)),
     ("rf", RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", n_jobs=-1, random_state=RS)),
     ("gb", HistGradientBoostingClassifier(class_weight="balanced", max_iter=300, random_state=RS))],
    voting="soft"))

In [ ]:
def evaluate(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return dict(Precision=precision_score(y_true, pred, zero_division=0),
                Recall=recall_score(y_true, pred),
                F1=f1_score(y_true, pred),
                ROC_AUC=roc_auc_score(y_true, proba),
                PR_AUC=average_precision_score(y_true, proba))

rows, fitted = [], {}
for name, m in models.items():
    t0 = time.time(); m.fit(X_train, y_train); tt = time.time() - t0
    proba = m.predict_proba(X_val)[:, 1]
    rows.append({"Model": name, **evaluate(y_val, proba), "Train_sec": tt})
    fitted[name] = m
val_table = pd.DataFrame(rows).set_index("Model").round(4)
print(val_table.sort_values("PR_AUC", ascending=False))
val_table.to_csv("validation_results.csv")

## 5. Pick best by **PR-AUC** on validation, tune the decision threshold on validation
Accuracy is useless here (99.8% by predicting "legit" always).

In [ ]:
best = val_table.drop(index="Baseline (majority)")["PR_AUC"].idxmax()
print("Best model:", best)
pv = fitted[best].predict_proba(X_val)[:, 1]
p, r, t = precision_recall_curve(y_val, pv)
f1s = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
thr = float(t[f1s.argmax()])
print(f"Chosen threshold (max F1 on validation): {thr:.3f}")

## 6. Final test evaluation (touched once)

In [ ]:
rows = []
for name, m in fitted.items():
    pt = m.predict_proba(X_test)[:, 1]
    rows.append({"Model": name, **evaluate(y_test, pt, thr if name == best else 0.5)})
test_table = pd.DataFrame(rows).set_index("Model").round(4)
print(test_table.sort_values("PR_AUC", ascending=False))
test_table.to_csv("test_results.csv")

pt = fitted[best].predict_proba(X_test)[:, 1]
pred = (pt >= thr).astype(int)
cm = confusion_matrix(y_test, pred); print(cm)
ConfusionMatrixDisplay(cm, display_labels=["Legit", "Fraud"]).plot(cmap="Blues")
plt.title(f"{best} - test set"); plt.savefig("confusion_matrix.png", dpi=120); plt.close()
p2, r2, _ = precision_recall_curve(y_test, pt)
plt.plot(r2, p2); plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("PR curve (test)")
plt.savefig("pr_curve.png", dpi=120); plt.close()

## 7. Error analysis & interpretation

In [ ]:
err = X_test.copy(); err["y"] = y_test.values; err["score"] = pt; err["pred"] = pred
fn = err[(err.y == 1) & (err.pred == 0)]; fp = err[(err.y == 0) & (err.pred == 1)]
print(f"Missed frauds (FN): {len(fn)}, avg amount {fn.Amount.mean():.2f}")
print(f"False alarms (FP): {len(fp)} -> {len(fp)/max(len(err[err.y==0]),1):.4%} of legit transactions")
rf = fitted["Random Forest"].named_steps["clf"]
cols = ["Time", "Amount"] + [c for c in X.columns if c not in ("Time", "Amount")]
imp = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False).head(10)
imp[::-1].plot.barh(title="Top-10 feature importance (Random Forest)")
plt.tight_layout(); plt.savefig("feature_importance.png", dpi=120); plt.close()
print(imp)

## 8. Save model for the prototype app

In [ ]:
joblib.dump({"model": fitted[best], "threshold": thr, "columns": list(X.columns), "name": best}, "fraud_model.joblib")
print("Saved fraud_model.joblib")